In [ ]:
!pip install langchain-community

In [ ]:
!pip install unstructured

In [ ]:
!pip install chromadb

In [51]:
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate

Load

In [14]:
DATA_PATH = "/content/books"

def load_document():
  loader = DirectoryLoader(DATA_PATH)
  documents = loader.load()
  print(type(documents))
  return documents

In [15]:
load_document()

<class 'list'>


[Document(metadata={'source': '/content/books/the_lost_key.markdown'}, page_content='# The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.\n\nAs dusk painted the sky in hues of amber, young Lila, a girl with eyes sharp as a hawk’s, spotted a glint by the riverbank. There, tangled in reeds, was the crescent-moon key, shim

Split

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 500,
    length_function = len,# what function can be used to measure the length of the text during split
    add_start_index = True,# start indedx is metadata field
)

chunks = text_splitter.split_documents(load_document)

TypeError: 'function' object is not iterable

In [19]:
def split_text(DATA_PATH):

  loader = DirectoryLoader(DATA_PATH)
  documents = loader.load()

  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 500,
    length_function = len,# what function can be used to measure the length of the text during split
    add_start_index = True,# start indedx is metadata field
  )

  chunks = text_splitter.split_documents(documents)
  return chunks


In [20]:
a_chunk = split_text(DATA_PATH)[0]

In [22]:
a_chunk.page_content

'# The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.'

In [23]:
a_chunk.metadata

{'source': '/content/books/the_lost_key.markdown', 'start_index': 0}

In [28]:
def create_vec_db(CHROMA_PATH, chunks):

  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

  db = Chroma.from_documents(
      chunks,
      embedding = embeddings,
      persist_directory=CHROMA_PATH

  )

In [26]:
# if need to erase the previous version of db
# if os.path.exists(_CHROMA_PATH):
#   shutil.rmtree(_CHROMA_PATH)

In [31]:
create_vec_db("/content/chroma", chunks=split_text(DATA_PATH))

In [34]:
em_func = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vec = em_func.embed_query("Apple fdsfds")

In [35]:
len(vec)

384

Retrieve

In [41]:
query_text = "What is the significance of the key in the story set in Elderglow?"

db = Chroma(persist_directory="/content/chroma", embedding_function=em_func)
result = db.similarity_search_with_relevance_scores(query_text, k = 3) #returns the best match of chunks

<ipython-input-41-5db043824fe1>:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'start_index': 0, 'source': '/content/books/the_lost_key.markdown'}, page_content='# The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.'), 0.4903988721726569), (Document(metadata={'source': '/content/books/

In [43]:
result# list of tuple

[(Document(metadata={'start_index': 0, 'source': '/content/books/the_lost_key.markdown'}, page_content='# The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.'),
  0.4903988721726569),
 (Document(metadata={'source': '/content/books/the_lost_key.markdown', 'start_index': 348}, page_content='One crisp autumn morning, Thane 

In [46]:
result[0][1]

0.4903988721726569

In [47]:
if len(result) == 0 or result[0][1] < 0.7: # setting the threshold score
  print("unable to fetch matching result")


unable to fetch matching result


Crafting Prompt

In [67]:
PROMPT_TEMPLATE ="""
Answer the question based only on the following context:
 {context}
Answer the question based on the above context: {query}
"""

In [68]:
context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in result])
prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
prompt = prompt_template.format_messages(
    context=context_text,
    query=query_text
)

In [69]:
context_text

'# The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.\n\n---\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders 

In [70]:
prompt

[HumanMessage(content=' \nAnswer the question based only on the following context:\n # The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.\n\n---\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, 

In [71]:
prompt[0].content

' \nAnswer the question based only on the following context:\n # The Lost Key\n\nIn the quiet village of Elderglow, nestled between whispering pines, lived an old locksmith named Thane. His hands, gnarled like the roots of the ancient oaks, had crafted keys for every door in the village. But one key, his own, was special—a silver piece etched with a crescent moon, said to unlock a chest of forgotten secrets.\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. The village noticed his distress, and soon, children and elders alike joined the hunt. They scoured cobbled paths, peeked into wells, and even asked the crows, who only cawed in mockery.\n\n---\n\nOne crisp autumn morning, Thane reached for the key, only to find it gone. Panic fluttered in his chest. He searched his cluttered shop, under iron tools and brass filings, but it was nowhere. Th